In [1]:
!pip install datasets trl bitsandbytes wandb


[notice] A new release of pip is available: 23.3.2 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Check CUDA availability and set device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# Load model and tokenizer directly to device
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B", torch_dtype=torch.float16, device_map={"": device})
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")

# Verify device
print(f"Model device: {next(model.parameters()).device}")

/opt/homebrew/Caskroom/miniconda/base/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model device: mps:0


In [3]:
import pandas as pd
from datasets import Dataset

# 1. Load the JSONL file as a Pandas DataFrame
file_path = '../datasets/translation_data_simple_seq2seq.jsonl'
df = pd.read_json(path_or_buf=file_path, lines=True)

# 2. Convert the Pandas DataFrame to a Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# 3. Define your formatting function (similar to formatting_prompts_func)
def formatting_func(examples):
    # Access the relevant columns from 'examples' and format them
    # For instance, if your JSONL has 'instruction', 'input', 'output' columns:
    inputs = examples["text"]
    outputs = examples["target"]

    # Apply your desired formatting logic, e.g., using a template:
    texts = [f"### Input: Translate from Javanese to Indonesian: {input}\n### Response: {output}"
             for input, output in zip(inputs, outputs)]

    return {"text": texts}

# 4. Apply the formatting function using dataset.map
dataset = dataset.map(formatting_func, batched=True)

Map: 100%|██████████| 500/500 [00:00<00:00, 68438.21 examples/s]


In [6]:
from trl import SFTTrainer
from transformers import TrainingArguments
import matplotlib.pyplot as plt

tokenizer.pad_token = tokenizer.eos_token

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,

    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 300,
        learning_rate = 1e-5,
        logging_steps = 1,
        optim = "adamw_torch",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
        use_mps_device=True
    ),
)

/var/folders/v8/kvhjgvyx233_y8gz7dxb4pbc0000gn/T/ipykernel_2179/1773626180.py:7: FutureWarning: `tokenizer` is deprecated and removed starting from version 0.16.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  trainer = SFTTrainer(
Tokenizing train dataset: 100%|██████████| 500/500 [00:00<00:00, 13539.45 examples/s]


In [7]:
trainer.train()

RuntimeError: MPS backend out of memory (MPS allocated: 13.30 GB, other allocations: 7.07 GB, max allowed: 20.40 GB). Tried to allocate 32.00 MB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [ ]:
# Plot loss after training
plt.figure(figsize=(10,5))
plt.plot(training_loss)
plt.title('Training Loss Over Time')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.grid(True)
plt.show()

In [ ]:
# Define your test input
test_input = "Cepak saka hotelku nginep, namung digawa mlaku, ing kene akeh tenan pilian panganane, panggonane sing amba, lan nyenengake."  # Example Javanese input

# Format the input using the same template as during training
formatted_input = f"### Input: Translate from Javanese to Indonesian: {test_input}\n### Response:"

# Tokenize the formatted input
input_ids = tokenizer(formatted_input, return_tensors="pt").input_ids.to(device)

with torch.no_grad():
       outputs = model.generate(input_ids=input_ids, attention_mask=input_ids.ne(tokenizer.pad_token_id), max_new_tokens=50)



# Decode the prediction
prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Print the prediction
print(prediction)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


### Input: Translate from Javanese to Indonesian: Cepak saka hotelku nginep, namung digawa mlaku, ing kene akeh tenan pilian panganane, panggonane sing amba, lan nyenengake.
### Response: Copak dari hotel saya menginap, namun digawa jalan, di sini banyak pilihan makanan, tempatnya yang luas, dan menyenangkan. 7 Response: Copak dari hotel saya menginap


In [ ]:
prediction

'### Input: Translate from Javanese to Indonesian: Cepak saka hotelku nginep, namung digawa mlaku, ing kene akeh tenan pilian panganane, panggonane sing amba, lan nyenengake.\n### Response: Copak dari hotel saya menginap, namun digawa jalan, di sini banyak pilihan makanan, tempatnya yang luas, dan menyenangkan. 7 Response: Copak dari hotel saya menginap'